In [ ]:
%run ./nb_silver_santos_avaliacao

In [1]:
import requests
import json
import pandas as pd
import numpy as np

df_avaliacoes = pd.read_parquet("/lakehouse/default/Files/silver/avaliacoes_servico/silver_avaliacoes_servico.parquet")

StatementMeta(, 38d52e29-8be2-4f4d-8b22-f9e95ec1a02c, 3, Finished, Available, Finished, False)

In [2]:
df_avaliacoes.columns = df_avaliacoes.columns.str.split("|").str[0]

def padronizar_servicos(df, coluna_alvo='nome_do_servico_avaliado'):
    """
    Altera os nomes dos serviços para a versão consolidada 
    com base no mapeamento de duplicatas e erros de grafia.
    """
    mapeamento = {
        "Agendamento de escoltas para cargas superdimensionadas": "Agendamento de escolta para carga superdimensionada",
        "Árvores - avaliação técnica": "Avaliação técnica de árvores",
        "Autorização temporária para carga e dlocais com restrições de estacionamento ou circulação": "Autorização temporária para carga e descarga em locais com restrições de estacionamento ou circulação",
        "Autorização temporária para carga e dscarga em locais com restrições de estacionamento ou circulação": "Autorização temporária para carga e descarga em locais com restrições de estacionamento ou circulação",
        "Fiscalização de veículos abandonados em via pública": "Fiscalização de veículo abandonado em via pública",
        "Instalação ou manutenção em rampa de acessibilidade - original": "Instalação ou manutenção em rampa de acessibilidade",
        "Instalação, manutenção e higienização de contentor": "Instalação, Manutenção ou Higienização de Contentor",
        "Instalação, manutenção ou higienização de contentor": "Instalação, Manutenção ou Higienização de Contentor",
        "Manutenção de calçadas de prédios públicos": "Manutenção em calçadas de prédios públicos",
        "Manutenção de guias ou do meio fio": "Manutenção de guias ou do meio-fio",
        "Manutenção em banheiro público": "Manutenção de banheiro público",
        "Manutenção em muretas dos canais - original": "Manutenção em muretas dos canais",
        "Mudan": "Mudança",
        "Serviço - poda de raiz de árvore": "Poda de raiz de árvore",
        "Tapa buraco": "Tapa-buraco",
        "Tapa buraco asfáltico": "Tapa-buraco"
    }

    # Aplica o mapeamento. Se o nome não estiver no dicionário, mantém o original.
    df[coluna_alvo] = df[coluna_alvo].replace(mapeamento)
    
    return df

df_avaliacoes = padronizar_servicos(df_avaliacoes)

df_avaliacoes['codFluxo'] = '12977'
df_avaliacoes['codCatalogo'] = '8225'
df_avaliacoes['dtAtualizacao'] = np.nan
df_avaliacoes['dataCriacao'] = np.nan
df_avaliacoes['codEtapa'] = 1
df_avaliacoes['etapa'] = "NA"
df_avaliacoes['executorAtual'] = "NA"
df_avaliacoes['seqEtapa'] = 1
df_avaliacoes['questao_resolvida'] = df_avaliacoes['questao_resolvida'].replace("", np.nan)

rename_map = {
    "n_da_solicitacao": "seqFluxo",
    "data_criacao": "dataSolicitacao",
    "data_finalizacao": "dataFinalizacao",
    "etapa": "etapa",
    "status_fluxo": "status",
    "detalhes": "detalhes_da_solicitacao",
    "email_interessado": "email_do_interessado",
    "observacoes_sugestoes_atendimento": "obs_classificacao_atendimento",
    "observacoes_sugestoes_servico_prestado": "obs_classificacao_de_servico_prestado",
}

df_avaliacoes = df_avaliacoes.rename(columns=rename_map)

ordem = [
    'seqFluxo', 'codFluxo', 'codCatalogo', 'servico', 'solicitante',
       'dataCriacao', 'dataSolicitacao', 'dtAtualizacao', 'dataFinalizacao',
       'codEtapa', 'etapa', 'status', 'executorAtual', 'seqEtapa', 'categoria',
       'classificacao_atendimento', 'classificacao_servico_prestado',
       'detalhes_da_solicitacao', 'email_do_interessado', 'expectativas',
       'nome_completo', 'nome_do_servico_avaliado',
       'obs_classificacao_atendimento',
       'obs_classificacao_de_servico_prestado', 'protocolo',
       'questao_resolvida', 'resposta_secretaria', 'area_responsavel'
]

df_avaliacoes = df_avaliacoes[ordem].copy()

df_avaliacoes = df_avaliacoes.astype(
    {
        "seqFluxo": "int",
    }
)

df_avaliacoes['dataSolicitacao'] = pd.to_datetime(df_avaliacoes['dataSolicitacao'], format='ISO8601').dt.date
df_avaliacoes['dataFinalizacao'] = pd.to_datetime(df_avaliacoes['dataFinalizacao'], format='ISO8601').dt.date

colunas_avaliacao = [
    'classificacao_atendimento', 'classificacao_servico_prestado', 
    'obs_classificacao_atendimento', 'obs_classificacao_de_servico_prestado'
]
for col in colunas_avaliacao:
    df_avaliacoes[col] = df_avaliacoes[col].replace("", np.nan)

StatementMeta(, 38d52e29-8be2-4f4d-8b22-f9e95ec1a02c, 4, Finished, Available, Finished, False)

# Remoção testes

In [3]:
os_sem_nome_servico = df_avaliacoes.query("nome_do_servico_avaliado == ''").sort_values(by="dataSolicitacao", ascending=False)
df_com_servico = df_avaliacoes.query("nome_do_servico_avaliado != ''").copy()
os_teste_resposta_secretaria = df_com_servico.loc[df_com_servico['resposta_secretaria'].str.contains('teste', na=False, case=False)].copy()
df_com_servico_sem_teste = df_com_servico.loc[~df_com_servico['resposta_secretaria'].str.contains('teste', na=False, case=False)].copy()
os_sem_protocolo = df_com_servico_sem_teste.loc[df_com_servico_sem_teste['protocolo'] == ''].copy()
df_com_servico_sem_teste_protocolo = df_com_servico_sem_teste.loc[df_com_servico_sem_teste['protocolo'] != ''].copy()
df_com_servico_sem_teste_protocolo['nome_do_servico_avaliado'] = df_com_servico_sem_teste_protocolo['nome_do_servico_avaliado'].str.capitalize()

StatementMeta(, 38d52e29-8be2-4f4d-8b22-f9e95ec1a02c, 5, Finished, Available, Finished, False)

# Tratamento area responsavel faltantes

In [4]:
dim_area_responsavel = df_com_servico_sem_teste_protocolo[['area_responsavel', 'nome_do_servico_avaliado']].drop_duplicates()

dim_area_responsavel = dim_area_responsavel.loc[~dim_area_responsavel['area_responsavel'].isin(['', '34', '40'])].copy()

# retira registros do serviço Fiscalização de veículo abandonado em via pública
# erroneamente vindo com area responsavel OUVIDORIA desde a origem
dim_area_responsavel = dim_area_responsavel.loc[
    ~((dim_area_responsavel['nome_do_servico_avaliado'] == 'Fiscalização de veículo abandonado em via pública')
    & (dim_area_responsavel['area_responsavel'] == 'OTC - OUVIDORIA, TRANSPARÊNCIA E CONTROLE'))
].copy()

df_com_servico_sem_teste_protocolo = df_com_servico_sem_teste_protocolo.drop(columns="area_responsavel")

df_com_servico_sem_teste_protocolo = df_com_servico_sem_teste_protocolo.merge(
    dim_area_responsavel, on="nome_do_servico_avaliado", how="left"
)
map_area_responsavel_faltantes = {
    "Adesão ao programa membership": "FPTS - FUNDAÇÃO PARQUE TECNOLÓGICO DE SANTOS",
    "Renovação de alvará de motorista auxiliar e motorista permissionário para autolotação": "CET - COMPANHIA DE ENGENHARIA DE TRÁFEGO",
    "Inscrição de condutor para motorista auxiliar de autolotação": "CET - COMPANHIA DE ENGENHARIA DE TRÁFEGO",
    "Árvores - avaliação técnica": "SEPREF - SECRETARIA DAS PREFEITURAS REGIONAIS",
    "Inscrição de condutor para motorista auxiliar para autolotação": "CET - COMPANHIA DE ENGENHARIA DE TRÁFEGO",
    "Instalação, manutenção e higienização de contentor": "SEGOV - SECRETARIA DE GOVERNO",
    "Autorização ou cancelamento para publicidade em táxi": "CET - COMPANHIA DE ENGENHARIA DE TRÁFEGO"
}

df_com_servico_sem_teste_protocolo['area_responsavel'] = df_com_servico_sem_teste_protocolo['area_responsavel'].fillna(
    df_com_servico_sem_teste_protocolo['nome_do_servico_avaliado'].map(map_area_responsavel_faltantes)
 )

StatementMeta(, 38d52e29-8be2-4f4d-8b22-f9e95ec1a02c, 6, Finished, Available, Finished, False)

In [11]:
df_avaliacoes_spark = spark.createDataFrame(df_com_servico_sem_teste_protocolo)

(
    df_avaliacoes_spark
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_avaliacoes_servico")
)

StatementMeta(, 38d52e29-8be2-4f4d-8b22-f9e95ec1a02c, 13, Finished, Available, Finished, False)